<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/OHDSI_Original25_GitHistory_Provenance_Recovery_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Clone the repository with full history

In [1]:
import sys, subprocess, os, json, re, sqlite3, gzip, tempfile, shutil, hashlib
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from IPython.display import display

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo-fullhistory")
PRIVATE_DIR = Path("/content/OHDSI_ORIGINAL25_GIT_RECOVERED_PRIVATE")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--no-single-branch", REPO_URL, str(REPO_DIR)],
    check=True
)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)

PRIVATE_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = REPO_DIR / "results" / "original25_git_history_recovery"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_DIR)
print("Private recovery dir:", PRIVATE_DIR)

Repository: /content/ohdsi-fhir-omop-showcase-demo-fullhistory
Private recovery dir: /content/OHDSI_ORIGINAL25_GIT_RECOVERED_PRIVATE


## 1. Reference signatures and helper functions

In [2]:
REF = {
    "person": 27,
    "visit_occurrence": 1386,
    "condition_occurrence": 983,
    "drug_exposure": 1275,
    "observation": 14168,
    "measurement": 14150,
}
REF_FHIR_PATIENTS = 25

SMALL_SIGNATURE_STRINGS = [
    "1386", "983", "1275", "14168", "14150", "78.12", "78.120", "0.00%",
    "25 synthetic", "27 person", "27 person records", "25 patients"
]

PROVENANCE_TERMS = [
    "synthea", "seed", "population", "patient.ndjson", "bulk",
    "pyomop", "sqlite", "create database", "create_db", "database",
    "unlink", "remove(", "drop table", "delete from", "append",
    "person_source_value", "visit_concept_id", "drug_concept_id"
]

def run_git(args, check=False):
    p = subprocess.run(
        ["git", "-C", str(REPO_DIR)] + args,
        capture_output=True, text=True
    )
    if check and p.returncode != 0:
        raise RuntimeError(p.stderr)
    return p

def table_exists(conn, table):
    return conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=? LIMIT 1",
        (table,)
    ).fetchone() is not None

def inspect_sqlite(path):
    row = {"path": str(path), "open_ok": False}
    try:
        conn = sqlite3.connect(f"file:{path}?mode=ro", uri=True, timeout=15)
        row["open_ok"] = True
        matches = 0
        for table, expected in REF.items():
            if table_exists(conn, table):
                observed = int(conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])
                row[table] = observed
                matches += int(observed == expected)
            else:
                row[table] = None
        row["small_signature_match_count"] = matches
        row["exact_small_signature"] = matches == len(REF)
        conn.close()
    except Exception as e:
        row["error"] = repr(e)
        row["small_signature_match_count"] = 0
        row["exact_small_signature"] = False
    return row

def count_fhir_patients(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    count = 0
    ids = set()
    bad = 0
    try:
        with opener(path, "rt", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    bad += 1
                    continue
                if isinstance(obj, dict) and obj.get("resourceType") == "Patient":
                    count += 1
                    if obj.get("id") is not None:
                        ids.add(str(obj["id"]))
                elif isinstance(obj, dict) and obj.get("resourceType") == "Bundle":
                    for entry in obj.get("entry", []) or []:
                        res = entry.get("resource", {})
                        if isinstance(res, dict) and res.get("resourceType") == "Patient":
                            count += 1
                            if res.get("id") is not None:
                                ids.add(str(res["id"]))
        return {
            "patient_resources": count,
            "unique_patient_ids": len(ids),
            "bad_json_lines": bad,
            "exact_25_patient_match": count == 25,
        }
    except Exception as e:
        return {
            "patient_resources": None,
            "unique_patient_ids": None,
            "bad_json_lines": None,
            "exact_25_patient_match": False,
            "error": repr(e),
        }

## 2. Git history inventory

In [3]:
log = run_git([
    "log", "--all", "--date=iso-strict",
    "--pretty=format:%H%x09%ad%x09%an%x09%s"
], check=True)

history_rows = []
for line in log.stdout.splitlines():
    parts = line.split("\t", 3)
    if len(parts) == 4:
        history_rows.append({
            "commit": parts[0],
            "date": parts[1],
            "author": parts[2],
            "subject": parts[3]
        })

history_df = pd.DataFrame(history_rows)
print("Commits found:", len(history_df))
display(history_df)
history_df.to_csv(RESULT_DIR / "git_commit_history.csv", index=False)

Commits found: 20


,commit,date,author,subject
0,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,SANGHATI BASU,Created using Colab
1,900b06b23117878728478b4a28ee0bf3454eea49,2026-08-15T22:12:19-05:00,SANGHATI BASU,Created using Colab
2,49b22d367ed162511fb7d888a4d70512085413f3,2026-08-15T19:04:52-05:00,SANGHATI BASU,Created using Colab
3,5a81d5e4fea849d93f7265100acd90e34462f60b,2026-08-15T18:50:15-05:00,SANGHATI BASU,Created using Colab
4,f493dcb5566f379eecebff5df0b856147aa4cbea,2026-08-15T17:47:41-05:00,SANGHATI BASU,Created using Colab
5,1fcb5d7e40da9368d3f5b532040aa39fe5d7a752,2026-08-15T16:55:42-05:00,SANGHATI BASU,Created using Colab
6,fc874f5c26040746b56bbd9b1622916ec7e4e8ef,2026-08-15T13:16:51-05:00,SANGHATI BASU,Created using Colab
7,35a913280f251fa1b8478c53f4f45a8aaad0b77e,2026-07-05T07:48:15+00:00,Sanghati Basu,Add FHIRpack framework comparator on MIMIC dem...
8,cf6530a3a1ffe4e2f9abb2cffa54898f04dc31e3,2026-07-05T05:48:17+00:00,Sanghati Basu,Add public deidentified MIMIC-IV FHIR validati...
9,0d80b213f092baf89202c84e370c553fc1d156a9,2026-07-05T03:41:26+00:00,Sanghati Basu,Add reviewer gap strengthening comparator and ...


## 3. Search all commits for the submitted small-cohort signature

This searches notebook source **and saved notebook outputs**, markdown, scripts, CSV/TXT/JSON, and shell files at each historical commit.

In [4]:
TEXT_EXTS = (
    ".ipynb", ".md", ".py", ".sh", ".txt", ".csv", ".json", ".yaml", ".yml"
)

signature_hits = []

commits = history_df["commit"].tolist()

for ci, commit in enumerate(commits, 1):
    print(f"[{ci}/{len(commits)}] scanning {commit[:10]}")
    tree = run_git(["ls-tree", "-r", "--name-only", commit])
    files = [x for x in tree.stdout.splitlines() if x.lower().endswith(TEXT_EXTS)]

    for fp in files:
        show = run_git(["show", f"{commit}:{fp}"])
        if show.returncode != 0:
            continue
        text = show.stdout
        low = text.lower()

        matched = []
        for term in SMALL_SIGNATURE_STRINGS:
            if term.lower() in low:
                matched.append(term)

        if matched:
            signature_hits.append({
                "commit": commit,
                "commit_date": history_df.loc[history_df["commit"] == commit, "date"].iloc[0],
                "file": fp,
                "matched_terms": " | ".join(matched),
                "match_count": len(matched),
            })

signature_hits_df = pd.DataFrame(signature_hits)
print("Signature-bearing historical files:", len(signature_hits_df))
if not signature_hits_df.empty:
    signature_hits_df = signature_hits_df.sort_values(
        ["match_count", "commit_date"], ascending=[False, True]
    )
    display(signature_hits_df.head(100))
    signature_hits_df.to_csv(
        RESULT_DIR / "small_cohort_signature_hits_by_commit.csv", index=False
    )

[1/20] scanning 138e6cde77
[2/20] scanning 900b06b231
[3/20] scanning 49b22d367e
[4/20] scanning 5a81d5e4fe
[5/20] scanning f493dcb556
[6/20] scanning 1fcb5d7e40
[7/20] scanning fc874f5c26
[8/20] scanning 35a913280f
[9/20] scanning cf6530a3a1
[10/20] scanning 0d80b213f0
[11/20] scanning a253888129
[12/20] scanning 567e5487f2
[13/20] scanning c4dcd73ef9
[14/20] scanning d8baf74f25
[15/20] scanning f08d8917b1
[16/20] scanning 43131773e3
[17/20] scanning 69d239187f
[18/20] scanning 11aa90c263
[19/20] scanning 07a57280d1
[20/20] scanning 35314736dd
Signature-bearing historical files: 219


,commit,commit_date,file,matched_terms,match_count
16,900b06b23117878728478b4a28ee0bf3454eea49,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,1386 | 983 | 1275 | 14168 | 14150 | 78.12 | 0....,8
2,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,1386 | 983 | 1275 | 14168 | 14150 | 78.12 | 0....,8
215,07a57280d10591352cd3690409b43740c740a982,2026-06-02T04:39:14-06:00,outputs/queries/vocabulary_mapping_summary.csv,1386 | 983 | 1275 | 14168 | 14150 | 78.12,6
211,11aa90c2632d58a8afda53f30124b9964cf39c80,2026-06-02T04:53:24-06:00,outputs/queries/vocabulary_mapping_summary.csv,1386 | 983 | 1275 | 14168 | 14150 | 78.12,6
205,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,outputs/queries/vocabulary_mapping_summary.csv,1386 | 983 | 1275 | 14168 | 14150 | 78.12,6
...,...,...,...,...,...
81,fc874f5c26040746b56bbd9b1622916ec7e4e8ef,2026-08-15T13:16:51-05:00,README.md,78.12 | 0.00%,2
90,fc874f5c26040746b56bbd9b1622916ec7e4e8ef,2026-08-15T13:16:51-05:00,results_reviewer_strengthening/fhir_omop_analy...,1386 | 983,2
69,1fcb5d7e40da9368d3f5b532040aa39fe5d7a752,2026-08-15T16:55:42-05:00,README.md,78.12 | 0.00%,2
78,1fcb5d7e40da9368d3f5b532040aa39fe5d7a752,2026-08-15T16:55:42-05:00,results_reviewer_strengthening/fhir_omop_analy...,1386 | 983,2


## 4. Extract contextual snippets around small-cohort evidence

The public report contains short snippets only. This is useful for determining whether the counts came from actual execution output, manuscript text, README text, or code constants.

In [5]:
snippet_rows = []

def clean_text_for_lines(text):
    # For ipynb JSON, raw lines are still useful because outputs/source are preserved.
    return text.splitlines()

for _, hit in signature_hits_df.head(200).iterrows() if not signature_hits_df.empty else []:
    commit = hit["commit"]
    fp = hit["file"]
    show = run_git(["show", f"{commit}:{fp}"])
    if show.returncode != 0:
        continue

    lines = clean_text_for_lines(show.stdout)
    for idx, line in enumerate(lines):
        low = line.lower()
        terms_here = [t for t in SMALL_SIGNATURE_STRINGS if t.lower() in low]
        if not terms_here:
            continue

        start = max(0, idx - 2)
        end = min(len(lines), idx + 3)
        context = "\n".join(lines[start:end])
        # Keep snippets bounded.
        context = context[:2000]

        snippet_rows.append({
            "commit": commit,
            "commit_date": hit["commit_date"],
            "file": fp,
            "line_index_approx": idx + 1,
            "terms": " | ".join(terms_here),
            "context": context,
        })

snippets_df = pd.DataFrame(snippet_rows)
print("Context snippets:", len(snippets_df))
if not snippets_df.empty:
    display(snippets_df[["commit_date","file","terms","context"]].head(50))
    snippets_df.to_csv(
        RESULT_DIR / "small_cohort_context_snippets.csv", index=False
    )

Context snippets: 617


,commit_date,file,terms,context
0,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,25 synthetic,"""\n"",\n ""### Reviewer-critical ..."
1,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,78.12,"""### Reviewer-critical questions\n"",\n..."
2,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,0.00%,"""1. Why did **25 synthetic FHIR Patien..."
3,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,0.00%,"""5. Can the workflow produce reproduci..."
4,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,983,"""| OMOP `person` | 27 |\n"",\n ""..."
5,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,0.00%,"""| `observation` | 14,168 |\n"",\n ..."
6,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,78.12,"""| `measurement` | 14,150 |\n"",\n ..."
7,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,0.00%,"""| Measurement mapping | 99.99% |\n"",\..."
8,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,1386,""" \""fhir_patient_resources\"": 25,\n..."
9,2026-08-15T22:12:19-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,983,""" \""person\"": 27,\n"",\n "" ..."


# Part A — Were the original database/FHIR files ever committed?

## 5. Search Git object history for SQLite and NDJSON filenames

Even if a file was later deleted, its blob can still exist in Git history.

In [6]:
all_objects = run_git(["rev-list", "--objects", "--all"], check=True)

historical_asset_rows = []
for line in all_objects.stdout.splitlines():
    parts = line.split(" ", 1)
    if len(parts) != 2:
        continue
    obj, fp = parts
    low = fp.lower()
    kind = None
    if low.endswith((".sqlite", ".sqlite3", ".db")):
        kind = "sqlite"
    elif low.endswith((".ndjson", ".jsonl", ".ndjson.gz", ".jsonl.gz")):
        kind = "fhir_ndjson"
    elif low.endswith((".zip", ".tar.gz", ".tgz")) and any(
        token in low for token in ["fhir", "omop", "synthea"]
    ):
        kind = "archive_candidate"

    if kind:
        historical_asset_rows.append({
            "object": obj,
            "path": fp,
            "kind": kind
        })

historical_assets_df = pd.DataFrame(historical_asset_rows)
print("Historical asset-like Git objects:", len(historical_assets_df))
if not historical_assets_df.empty:
    display(historical_assets_df)
    historical_assets_df.to_csv(
        RESULT_DIR / "historical_asset_objects.csv", index=False
    )

Historical asset-like Git objects: 0


## 6. Recover and inspect historical SQLite blobs privately

If an SQLite blob ever existed in Git, this cell writes it only to the private Colab directory and checks its OMOP row counts.

In [7]:
historical_db_results = []

if not historical_assets_df.empty:
    db_objs = historical_assets_df[historical_assets_df["kind"] == "sqlite"]
else:
    db_objs = pd.DataFrame()

for idx, row in db_objs.iterrows():
    obj = row["object"]
    original_path = row["path"]
    target = PRIVATE_DIR / f"gitblob_{obj[:12]}_{Path(original_path).name}"

    p = subprocess.run(
        ["git", "-C", str(REPO_DIR), "cat-file", "-p", obj],
        capture_output=True
    )
    if p.returncode != 0:
        continue
    target.write_bytes(p.stdout)

    result = inspect_sqlite(target)
    result["git_object"] = obj
    result["historical_path"] = original_path
    historical_db_results.append(result)

historical_db_df = pd.DataFrame(historical_db_results)
print("Historical SQLite blobs inspected:", len(historical_db_df))
if not historical_db_df.empty:
    historical_db_df = historical_db_df.sort_values(
        ["exact_small_signature", "small_signature_match_count"],
        ascending=False
    )
    display(historical_db_df)
    historical_db_df.drop(columns=["path"], errors="ignore").to_csv(
        RESULT_DIR / "historical_sqlite_blob_results.csv", index=False
    )

Historical SQLite blobs inspected: 0


## 7. Recover and inspect historical FHIR Patient NDJSON blobs privately

Only files with `patient` in the historical filename are parsed.

In [8]:
historical_patient_results = []

if not historical_assets_df.empty:
    fhir_objs = historical_assets_df[
        (historical_assets_df["kind"] == "fhir_ndjson") &
        (historical_assets_df["path"].str.lower().str.contains("patient"))
    ]
else:
    fhir_objs = pd.DataFrame()

for idx, row in fhir_objs.iterrows():
    obj = row["object"]
    original_path = row["path"]
    suffix = ".ndjson.gz" if original_path.lower().endswith(".gz") else ".ndjson"
    target = PRIVATE_DIR / f"gitblob_{obj[:12]}_Patient{suffix}"

    p = subprocess.run(
        ["git", "-C", str(REPO_DIR), "cat-file", "-p", obj],
        capture_output=True
    )
    if p.returncode != 0:
        continue
    target.write_bytes(p.stdout)

    result = count_fhir_patients(target)
    result["git_object"] = obj
    result["historical_path"] = original_path
    historical_patient_results.append(result)

historical_patient_df = pd.DataFrame(historical_patient_results)
print("Historical Patient blobs inspected:", len(historical_patient_df))
if not historical_patient_df.empty:
    historical_patient_df = historical_patient_df.sort_values(
        ["exact_25_patient_match", "patient_resources"],
        ascending=False
    )
    display(historical_patient_df)
    historical_patient_df.to_csv(
        RESULT_DIR / "historical_patient_blob_results.csv", index=False
    )

Historical Patient blobs inspected: 0


# Part B — Recover generation/import provenance

## 8. Search every historical text file for Synthea, seed, database lifecycle, and import clues

In [9]:
prov_hits = []

for ci, commit in enumerate(commits, 1):
    tree = run_git(["ls-tree", "-r", "--name-only", commit])
    files = [x for x in tree.stdout.splitlines() if x.lower().endswith(TEXT_EXTS)]

    for fp in files:
        show = run_git(["show", f"{commit}:{fp}"])
        if show.returncode != 0:
            continue
        lines = show.stdout.splitlines()

        for i, line in enumerate(lines, 1):
            low = line.lower()
            terms = [t for t in PROVENANCE_TERMS if t.lower() in low]
            if not terms:
                continue

            prov_hits.append({
                "commit": commit,
                "commit_date": history_df.loc[history_df["commit"] == commit, "date"].iloc[0],
                "file": fp,
                "line": i,
                "terms": " | ".join(terms),
                "text": line[:1500]
            })

prov_df = pd.DataFrame(prov_hits)
print("Provenance-line hits:", len(prov_df))
if not prov_df.empty:
    display(prov_df.head(100))
    prov_df.to_csv(
        RESULT_DIR / "historical_provenance_keyword_hits.csv", index=False
    )

Provenance-line hits: 6051


,commit,commit_date,file,line,terms,text
0,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,10,pyomop,"""<a href=\""https://colab.research.goog..."
1,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,20,pyomop,"""# FHIRy–pyOMOP Transformation Fidelit..."
2,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,22,pyomop,"""FHIR R4 → FHIRy → pandas → pyOMOP → O..."
3,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,68,pyomop,"""WARNING:pyomop:WARNING: LLM is no..."
4,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,76,pyomop,"""pyomop: 6.4.0\n"","
...,...,...,...,...,...,...
95,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,4280,append,""" statuses.append(F..."
96,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,4282,append,""" warnings.append(\..."
97,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,4283,append,""" statuses.append(F..."
98,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,4299,append,""" warnings.append(\..."


## 9. Isolate likely Synthea generation commands and population/seed settings

The goal is to find a reproducible historical command such as a population size, seed, output mode, or fixed version.

In [10]:
generation_rows = []

if not prov_df.empty:
    for _, r in prov_df.iterrows():
        text = str(r["text"])
        low = text.lower()

        score = 0
        reasons = []

        if "synthea" in low:
            score += 5; reasons.append("synthea")
        if "seed" in low:
            score += 4; reasons.append("seed")
        if re.search(r"(^|\D)25(\D|$)", text):
            score += 3; reasons.append("contains_25")
        if any(tok in low for tok in ["-p ", "--population", "population="]):
            score += 4; reasons.append("population_argument")
        if any(tok in low for tok in ["fhir", "ndjson", "bulk"]):
            score += 2; reasons.append("fhir_output")
        if any(tok in low for tok in ["run_synthea", "./run_synthea", "java -jar"]):
            score += 3; reasons.append("executable_command")

        if score > 0:
            generation_rows.append({
                "commit": r["commit"],
                "commit_date": r["commit_date"],
                "file": r["file"],
                "line": r["line"],
                "score": score,
                "reasons": " | ".join(reasons),
                "text": text
            })

generation_df = pd.DataFrame(generation_rows)
if not generation_df.empty:
    generation_df = generation_df.sort_values(
        ["score", "commit_date"], ascending=[False, True]
    )
    display(generation_df.head(100))
    generation_df.to_csv(
        RESULT_DIR / "synthea_generation_command_candidates.csv", index=False
    )
else:
    print("No strong historical Synthea generation command candidates found.")

,commit,commit_date,file,line,score,reasons,text
1800,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,scripts/09_create_showcase_figure.py,55,8,synthea | contains_25,"box(0.4, 4.8, 1.7, 0.9, ""Synthea\nSynthetic Pa..."
1761,43131773e371df52121aedcf280f6428ac230cdb,2026-06-04T10:30:41-05:00,scripts/09_create_showcase_figure.py,55,8,synthea | contains_25,"box(0.4, 4.8, 1.7, 0.9, ""Synthea\nSynthetic Pa..."
1677,d8baf74f259fb35ac108466ea58afbb23052e5a6,2026-06-10T08:16:37-05:00,scripts/09_create_showcase_figure.py,55,8,synthea | contains_25,"box(0.4, 4.8, 1.7, 0.9, ""Synthea\nSynthetic Pa..."
1722,f08d8917b15b6961842421a4d0494a6aefab000c,2026-06-10T12:41:06+00:00,scripts/09_create_showcase_figure.py,55,8,synthea | contains_25,"box(0.4, 4.8, 1.7, 0.9, ""Synthea\nSynthetic Pa..."
1636,c4dcd73ef97241cf271636427e5eaba465ca0185,2026-07-04T23:24:38+00:00,scripts/09_create_showcase_figure.py,55,8,synthea | contains_25,"box(0.4, 4.8, 1.7, 0.9, ""Synthea\nSynthetic Pa..."
...,...,...,...,...,...,...,...
1884,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,outputs/queries/ohdsi_demo_narration_script.md,13,5,synthea,"First, I generate synthetic patient data using..."
1849,07a57280d10591352cd3690409b43740c740a982,2026-06-02T04:39:14-06:00,outputs/queries/ohdsi_demo_asset_checklist.md,48,5,synthea,Synthetic patients were generated using Synthea.
1852,07a57280d10591352cd3690409b43740c740a982,2026-06-02T04:39:14-06:00,outputs/queries/ohdsi_demo_narration_script.md,13,5,synthea,"First, I generate synthetic patient data using..."
1817,11aa90c2632d58a8afda53f30124b9964cf39c80,2026-06-02T04:53:24-06:00,outputs/queries/ohdsi_demo_asset_checklist.md,48,5,synthea,Synthetic patients were generated using Synthea.


## 10. Audit historical database lifecycle logic

This checks whether historical code explicitly removed/recreated the SQLite database before import, or whether reuse/append behavior is plausible from the preserved code.

**Important:** A plausible append path is not itself proof that it caused the 25→27 discrepancy.

In [11]:
lifecycle_terms = [
    "unlink", "os.remove", "remove(", "drop table", "delete from",
    "create_all", "create database", "sqlite3.connect", "session.add",
    "bulk_save", "append", "to_sql", "if db", "exists()"
]

lifecycle_rows = []

if not prov_df.empty:
    for _, r in prov_df.iterrows():
        low = str(r["text"]).lower()
        matched = [t for t in lifecycle_terms if t in low]
        if matched:
            lifecycle_rows.append({
                "commit": r["commit"],
                "commit_date": r["commit_date"],
                "file": r["file"],
                "line": r["line"],
                "matched_lifecycle_terms": " | ".join(matched),
                "text": r["text"]
            })

lifecycle_df = pd.DataFrame(lifecycle_rows)
print("Database lifecycle evidence lines:", len(lifecycle_df))
if not lifecycle_df.empty:
    display(lifecycle_df.head(150))
    lifecycle_df.to_csv(
        RESULT_DIR / "database_lifecycle_evidence.csv", index=False
    )

Database lifecycle evidence lines: 1621


,commit,commit_date,file,line,matched_lifecycle_terms,text
0,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,304,append,""" element.appendChild(doc..."
1,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,420,append,""" files.append(..."
2,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,480,append,""" candidates.append((score, p))..."
3,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,508,append,""" DB_RESOLUTION_ROWS.append({\n"","
4,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,779,append,""" element.appendChild(doc..."
...,...,...,...,...,...,...
145,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Execution_v8.ipynb,8089,sqlite3.connect,""" with sqlite3.connect(db_path) as ..."
146,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Execution_v8.ipynb,8109,append,""" cohort_rows.append({\n"","
147,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Execution_v8.ipynb,8476,append,""" element.appendChild(doc..."
148,138e6cde779e7170f2c12d2134f470f4bdb36711,2026-08-15T22:20:45-05:00,FHIRy_pyOMOP_TFL_Execution_v8.ipynb,8558,append,""" prevalence_rows.append({\n"","


# Part C — Find exact notebook outputs that may preserve the 25→27 event

## 11. Parse historical notebooks structurally

This cell looks inside notebook JSON cells/outputs for combinations of:

- `25`
- `27`
- `person`
- row-count signatures
- mapping percentages

It records the cell source and output separately so manuscript text is not confused with execution evidence.

In [12]:
notebook_evidence = []

for commit in commits:
    tree = run_git(["ls-tree", "-r", "--name-only", commit])
    notebooks = [x for x in tree.stdout.splitlines() if x.lower().endswith(".ipynb")]

    for fp in notebooks:
        show = run_git(["show", f"{commit}:{fp}"])
        if show.returncode != 0:
            continue
        try:
            obj = json.loads(show.stdout)
        except Exception:
            continue

        for cell_idx, cell in enumerate(obj.get("cells", [])):
            source = "".join(cell.get("source", []))
            output_texts = []

            for output in cell.get("outputs", []) or []:
                if "text" in output:
                    t = output["text"]
                    output_texts.append("".join(t) if isinstance(t, list) else str(t))
                data = output.get("data", {})
                for key in ["text/plain", "text/markdown"]:
                    if key in data:
                        t = data[key]
                        output_texts.append("".join(t) if isinstance(t, list) else str(t))

            output_text = "\n".join(output_texts)
            combined = source + "\n" + output_text
            low = combined.lower()

            evidence_terms = []
            for term in ["25", "27", "1386", "983", "1275", "14168", "14150", "78.12", "visit_concept_id", "drug_concept_id"]:
                if term.lower() in low:
                    evidence_terms.append(term)

            if len(evidence_terms) >= 2 and ("person" in low or "omop" in low or "mapping" in low):
                notebook_evidence.append({
                    "commit": commit,
                    "commit_date": history_df.loc[history_df["commit"] == commit, "date"].iloc[0],
                    "file": fp,
                    "cell_index": cell_idx,
                    "cell_type": cell.get("cell_type"),
                    "execution_count": cell.get("execution_count"),
                    "evidence_terms": " | ".join(evidence_terms),
                    "source_excerpt": source[:2500],
                    "output_excerpt": output_text[:4000],
                })

notebook_evidence_df = pd.DataFrame(notebook_evidence)
print("Historical notebook evidence cells:", len(notebook_evidence_df))
if not notebook_evidence_df.empty:
    notebook_evidence_df = notebook_evidence_df.sort_values(
        ["commit_date", "file", "cell_index"]
    )
    display(
        notebook_evidence_df[
            ["commit_date","file","cell_index","execution_count","evidence_terms","output_excerpt"]
        ].head(100)
    )
    notebook_evidence_df.to_csv(
        RESULT_DIR / "historical_notebook_execution_evidence.csv", index=False
    )

Historical notebook evidence cells: 73


,commit_date,file,cell_index,execution_count,evidence_terms,output_excerpt
71,2026-08-15T13:16:51-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,5,2.0,25 | 27,Mounted at /content/drive\nNon-empty database ...
72,2026-08-15T13:16:51-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,13,7.0,25 | 27 | visit_concept_id | drug_concept_id,table total_rows mapped_co...
65,2026-08-15T16:55:42-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,5,2.0,25 | 27,Mounted at /content/drive\nNon-empty database ...
66,2026-08-15T16:55:42-05:00,FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb,13,7.0,25 | 27 | visit_concept_id | drug_concept_id,table total_rows mapped_co...
67,2026-08-15T16:55:42-05:00,FHIRy_pyOMOP_TFL_Execution_v8.ipynb,13,7.0,25 | 27,\n ===========================================...
...,...,...,...,...,...,...
19,2026-08-15T22:20:45-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,4,2.0,25 | 27 | 1386 | 983 | 1275 | 14168 | 14150 | ...,Mounted at /content/drive\nDATA_ROOT: /content...
20,2026-08-15T22:20:45-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,9,NaN,25 | 27,
21,2026-08-15T22:20:45-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,16,8.0,visit_concept_id | drug_concept_id,Empty DataFrame\nColumns: []\nIndex: []
22,2026-08-15T22:20:45-05:00,OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb,22,11.0,25 | 27 | 983 | visit_concept_id | drug_concep...,WARNING:pyomop:WARNING: LLM is not installed. ...


## 12. Build a chronological provenance timeline

This combines:
- commit history,
- small-cohort signature hits,
- generation-command candidates,
- notebook execution evidence.

In [13]:
timeline_rows = []

if not signature_hits_df.empty:
    for _, r in signature_hits_df.iterrows():
        timeline_rows.append({
            "date": r["commit_date"],
            "commit": r["commit"],
            "file": r["file"],
            "evidence_type": "small_cohort_signature",
            "detail": r["matched_terms"]
        })

if not generation_df.empty:
    for _, r in generation_df.head(200).iterrows():
        timeline_rows.append({
            "date": r["commit_date"],
            "commit": r["commit"],
            "file": r["file"],
            "evidence_type": "generation_or_seed_candidate",
            "detail": r["text"][:1000]
        })

if not notebook_evidence_df.empty:
    for _, r in notebook_evidence_df.head(300).iterrows():
        timeline_rows.append({
            "date": r["commit_date"],
            "commit": r["commit"],
            "file": r["file"],
            "evidence_type": "notebook_execution_evidence",
            "detail": (r["evidence_terms"] + " | " + str(r["output_excerpt"])[:1200])
        })

timeline_df = pd.DataFrame(timeline_rows)
if not timeline_df.empty:
    timeline_df = timeline_df.sort_values(["date","file"])
    display(timeline_df.head(200))
    timeline_df.to_csv(RESULT_DIR / "original25_provenance_timeline.csv", index=False)

,date,commit,file,evidence_type,detail
236,2026-05-28T20:16:05-05:00,35314736dd3e5959e53660ae07326c4327e59dbf,README.md,generation_or_seed_candidate,1. Generation of synthetic FHIR Bulk NDJSON da...
237,2026-05-28T20:16:05-05:00,35314736dd3e5959e53660ae07326c4327e59dbf,README.md,generation_or_seed_candidate,The workflow begins with synthetic patient dat...
238,2026-05-28T20:16:05-05:00,35314736dd3e5959e53660ae07326c4327e59dbf,README.md,generation_or_seed_candidate,1. Generate synthetic FHIR Bulk NDJSON records...
109,2026-05-28T20:16:05-05:00,35314736dd3e5959e53660ae07326c4327e59dbf,outputs/queries/condition_prevalence_top15.csv,small_cohort_signature,983
313,2026-05-28T20:16:05-05:00,35314736dd3e5959e53660ae07326c4327e59dbf,outputs/queries/ohdsi_demo_asset_checklist.md,generation_or_seed_candidate,Synthetic patients were generated using Synthea.
...,...,...,...,...,...
68,2026-07-05T01:51:38+00:00,a253888129ec423ead1f3404c4a2469ea6fea20f,scripts/09_create_showcase_figure.py,small_cohort_signature,983 | 78.12 | 0.00%
225,2026-07-05T01:51:38+00:00,a253888129ec423ead1f3404c4a2469ea6fea20f,scripts/09_create_showcase_figure.py,generation_or_seed_candidate,"box(0.4, 4.8, 1.7, 0.9, ""Synthea\nSynthetic Pa..."
88,2026-07-05T03:41:26+00:00,0d80b213f092baf89202c84e370c553fc1d156a9,README.md,small_cohort_signature,78.12 | 0.00%
278,2026-07-05T03:41:26+00:00,0d80b213f092baf89202c84e370c553fc1d156a9,README.md,generation_or_seed_candidate,* Synthetic FHIR Bulk NDJSON data generation u...


# Part D — Decision logic

## 13. Determine what was actually recovered

Possible outcomes:

- **EXACT_ASSETS_RECOVERED_FROM_GIT** — exact historical SQLite and/or 25-patient FHIR blobs existed in Git.
- **EXECUTION_PROVENANCE_RECOVERED** — no raw assets, but historical notebook execution evidence preserves the small-cohort event strongly enough for deeper diagnosis.
- **GENERATION_PROVENANCE_RECOVERED** — a reproducible historical Synthea command/seed was found.
- **PROVENANCE_INCOMPLETE** — history still does not support a defensible explanation.

In [14]:
exact_db_git = (
    historical_db_df[historical_db_df["exact_small_signature"] == True]
    if not historical_db_df.empty else pd.DataFrame()
)
exact_fhir_git = (
    historical_patient_df[historical_patient_df["exact_25_patient_match"] == True]
    if not historical_patient_df.empty else pd.DataFrame()
)

strong_exec = pd.DataFrame()
if not notebook_evidence_df.empty:
    strong_exec = notebook_evidence_df[
        notebook_evidence_df["evidence_terms"].str.contains("1386|1275|14168|14150|78.12", regex=True)
    ]

strong_generation = pd.DataFrame()
if not generation_df.empty:
    strong_generation = generation_df[
        generation_df["score"] >= 7
    ]

if len(exact_db_git) > 0 or len(exact_fhir_git) > 0:
    recovery_status = "EXACT_ASSETS_RECOVERED_FROM_GIT"
elif len(strong_exec) > 0:
    recovery_status = "EXECUTION_PROVENANCE_RECOVERED"
elif len(strong_generation) > 0:
    recovery_status = "GENERATION_PROVENANCE_RECOVERED"
else:
    recovery_status = "PROVENANCE_INCOMPLETE"

status_df = pd.DataFrame([{
    "status": recovery_status,
    "exact_sqlite_blobs": len(exact_db_git),
    "exact_25_patient_fhir_blobs": len(exact_fhir_git),
    "strong_execution_evidence_cells": len(strong_exec),
    "strong_generation_candidates": len(strong_generation),
    "root_cause_claim_allowed": False,
    "note": (
        "A root cause still requires direct evidence; recovery status alone is not a causal conclusion."
    )
}])

display(status_df)
status_df.to_csv(RESULT_DIR / "git_history_recovery_status.csv", index=False)

print("\n" + "="*78)
print("GIT-HISTORY RECOVERY STATUS:", recovery_status)
print("="*78)

,status,exact_sqlite_blobs,exact_25_patient_fhir_blobs,strong_execution_evidence_cells,strong_generation_candidates,root_cause_claim_allowed,note
0,EXECUTION_PROVENANCE_RECOVERED,0,0,6,94,False,A root cause still requires direct evidence; r...



GIT-HISTORY RECOVERY STATUS: EXECUTION_PROVENANCE_RECOVERED


## 14. Evidence-based 25→27 interpretation gate

This cell does **not** automatically claim that the two extra PERSON rows were stale/duplicate records.

It produces one of three conclusions:

1. **Directly testable** — exact DB + exact FHIR recovered.
2. **Historically evidenced but not directly testable** — saved execution outputs exist, but row-level assets are gone.
3. **Unresolved** — insufficient preserved evidence.

That distinction is important for a reviewer response.

In [15]:
if len(exact_db_git) > 0 and len(exact_fhir_git) > 0:
    interpretation_gate = (
        "DIRECTLY_TESTABLE: exact historical database and 25-patient FHIR asset were recovered from Git history. "
        "Proceed to record-level reconciliation before stating a cause."
    )
elif len(strong_exec) > 0:
    interpretation_gate = (
        "HISTORICALLY_EVIDENCED_NOT_DIRECTLY_TESTABLE: historical notebook execution evidence preserves the small-cohort "
        "counts, but exact row-level source/target assets are not both available. Do not state a causal explanation yet."
    )
else:
    interpretation_gate = (
        "UNRESOLVED: preserved Git/Drive evidence is insufficient to explain the historical 25-to-27 discrepancy. "
        "A clean, explicitly labeled reproduction run is the scientifically defensible next step."
    )

print(interpretation_gate)

gate_df = pd.DataFrame([{
    "interpretation_gate": interpretation_gate
}])
gate_df.to_csv(RESULT_DIR / "person_25_to_27_interpretation_gate.csv", index=False)

HISTORICALLY_EVIDENCED_NOT_DIRECTLY_TESTABLE: historical notebook execution evidence preserves the small-cohort counts, but exact row-level source/target assets are not both available. Do not state a causal explanation yet.


## 15. Generate a concise provenance report

In [16]:
report = [
    "# OHDSI Original 25-Patient Provenance Recovery Report",
    "",
    f"**Recovery status:** {recovery_status}",
    "",
    "## What was checked",
    "",
    f"- Git commits scanned: {len(history_df)}",
    f"- Historical small-cohort signature files: {len(signature_hits_df)}",
    f"- Historical SQLite blobs: {len(historical_db_df)}",
    f"- Historical Patient NDJSON blobs: {len(historical_patient_df)}",
    f"- Strong historical execution-evidence cells: {len(strong_exec)}",
    f"- Strong generation/seed command candidates: {len(strong_generation)}",
    "",
    "## 25→27 interpretation",
    "",
    interpretation_gate,
    "",
    "## Scientific constraint",
    "",
    "No stale-row, duplicate-import, pyOMOP bug, or Synthea explanation should be written into the manuscript "
    "unless the preserved evidence directly supports it.",
]

report_path = RESULT_DIR / "OHDSI_ORIGINAL25_GIT_PROVENANCE_REPORT.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print(report_path.read_text(encoding="utf-8"))

# OHDSI Original 25-Patient Provenance Recovery Report

**Recovery status:** EXECUTION_PROVENANCE_RECOVERED

## What was checked

- Git commits scanned: 20
- Historical small-cohort signature files: 219
- Historical SQLite blobs: 0
- Historical Patient NDJSON blobs: 0
- Strong historical execution-evidence cells: 6
- Strong generation/seed command candidates: 94

## 25→27 interpretation

HISTORICALLY_EVIDENCED_NOT_DIRECTLY_TESTABLE: historical notebook execution evidence preserves the small-cohort counts, but exact row-level source/target assets are not both available. Do not state a causal explanation yet.

## Scientific constraint

No stale-row, duplicate-import, pyOMOP bug, or Synthea explanation should be written into the manuscript unless the preserved evidence directly supports it.


## 16. Public vs private outputs

GitHub-safe outputs are under:

```text
results/original25_git_history_recovery/
```

Historical SQLite/FHIR blobs, if any were recovered, remain under:

```text
/content/OHDSI_ORIGINAL25_GIT_RECOVERED_PRIVATE/
```

**Do not push the private recovered blobs automatically.**

After this notebook is run and saved, push the executed notebook to GitHub and share its link. The next step will depend on the evidence status:

- exact assets → direct reviewer diagnosis,
- execution provenance only → historical-evidence audit,
- incomplete provenance → clean labeled reproduction run with fresh-database assertions and timing.